## Estimate y=mx+q between two voxels on Harald and plot the m map 

In [1]:
import os
import nibabel as nib
import numpy as np
import json
from datetime import datetime

home = r"/home/malberti/wks14/temp/FF_DWI_Drift"
derivatives = os.path.join(home, "derivatives")

vp = 15
metric = "MD"
sessions = ["ses-01","ses-03","ses-04","ses-05",
            "ses-06","ses-07","ses-08","ses-09","ses-10",
            "ses-11","ses-12","ses-13", "ses-14","ses-15","ses-16", "ses-17", 
            "ses-18","ses-19", "ses-20", "ses-21","ses-22" ]

#sessions = ["ses-02","ses-06","ses-10","ses-13"]

id = f"sub-{vp}"

for i in range(len(sessions) - 1):  # avoid overflow

    ses1 = sessions[i]
    ses2 = sessions[i+1]

    # ---- Load MD maps ----
    img1 = nib.load(os.path.join(
        derivatives, id, ses1, "dwi", "index", "Tensor",
        f"{id}_{ses1}_prep_{metric}.nii"))

    img2 = nib.load(os.path.join(
        derivatives, id, ses2, "dwi", "index", "Tensor",
        f"{id}_{ses2}_prep_{metric}.nii"))

    md1 = img1.get_fdata()
    md2 = img2.get_fdata()

    # ---- Load acquisition times ----
    json1 = os.path.join(home, "rawdata", id, ses1, "dwi",
                         f"{id}_{ses1}_dir-AP_part-mag_dwi.json")

    json2 = os.path.join(home, "rawdata", id, ses2, "dwi",
                         f"{id}_{ses2}_dir-AP_part-mag_dwi.json")

    with open(json1, "r") as f:
        meta1 = json.load(f)

    with open(json2, "r") as f:
        meta2 = json.load(f)

    t1_str = meta1.get("AcquisitionTime")
    t2_str = meta2.get("AcquisitionTime")

    # convert HH:MM:SS.sss to seconds
    t1 = datetime.strptime(t1_str, "%H:%M:%S.%f")
    t2 = datetime.strptime(t2_str, "%H:%M:%S.%f")

    delta_t = (t1 - t2).total_seconds()

    if delta_t == 0:
        print(f"Skipping {ses1}-{ses2}: zero time difference")
        continue

    # ---- Compute angular coefficient voxelwise ----
    coeff = ((md1 - md2) / delta_t)

    # ---- Save as NIfTI ----
    out_img = nib.Nifti1Image(coeff, affine=img1.affine, header=img1.header)

    out_path = os.path.join(
        derivatives, id,
        f"{id}_{ses1}_to_{ses2}_{metric}_drift_coeff.nii.gz"
    )

    nib.save(out_img, out_path)

    print(f"Saved: {out_path}")

Saved: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-15/sub-15_ses-01_to_ses-03_MD_drift_coeff.nii.gz
Saved: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-15/sub-15_ses-03_to_ses-04_MD_drift_coeff.nii.gz
Saved: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-15/sub-15_ses-04_to_ses-05_MD_drift_coeff.nii.gz
Saved: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-15/sub-15_ses-05_to_ses-06_MD_drift_coeff.nii.gz
Saved: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-15/sub-15_ses-06_to_ses-07_MD_drift_coeff.nii.gz
Saved: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-15/sub-15_ses-07_to_ses-08_MD_drift_coeff.nii.gz
Saved: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-15/sub-15_ses-08_to_ses-09_MD_drift_coeff.nii.gz
Saved: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-15/sub-15_ses-09_to_ses-10_MD_drift_coeff.nii.gz
Saved: /home/malberti/wks14/temp/FF_DWI_Drift/derivatives/sub-15/sub-15_ses-10_to_ses-11_MD_drift_coeff.nii.gz
S

# Same coeff. estimation on set A of SWEEP2.  
I again choose the delta-t based on acqusition time on the Json file 

In [ ]:
import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import json
from datetime import datetime

#I used the Json time stamp to have a better approx. 


sessions=["ses-01","ses-02"]
sets=["A", "B", "C"]
metric="MD" #"FA", 

derivatives=r"/home/malberti/Unix_Folders/SWEEP2/DEWEY/derivatives"  # sub-200\ses-01\dwi\ms_analysis\index2MNI\smooth\sub-00_ses-01_prep2T1w_MDk2MNI6mm.nii.gz""
rawdata = r"/home/malberti/Unix_Folders/SWEEP2/DEWEY/rawdata" 

vps= [
    0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16,
    17, 18, 19, 20, 21, 22, 25, 26, 27, 28, 29, 30, 31,
    33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43
]


#vps = [1, 3, 5, 10, 12, 14, 16, 19, 25, 27, 30, 36, 38, 42] #1
#vps = [2, 4, 6, 11, 13, 15, 17, 20, 26, 28, 31, 33, 37, 39, 43]

kernel="6mm"

set="A"
mean_coeffs = []


for vp in vps:

    id = f"sub-{vp:02d}"
    project_id = f"sub-2{vp:02d}"

    ses1 = "ses-01"
    ses2 = "ses-02"

    # ---- Load MD maps ----
    img1_path = os.path.join(
        derivatives, project_id, ses1,
        "dwi/index2MNI/smooth",
        f"{id}_{ses1}_prep_{set}_{metric}2MNI_{kernel}.nii.gz"
    )

    img2_path = os.path.join(
        derivatives, project_id, ses2,
        "dwi/index2MNI/smooth",
        f"{id}_{ses2}_prep_{set}_{metric}2MNI_{kernel}.nii.gz"
    )



    img1 = nib.load(img1_path)
    img2 = nib.load(img2_path)

    data1 = img1.get_fdata()
    data2 = img2.get_fdata()

    # ---- Load acquisition times ----
    json1 = os.path.join(
        rawdata, project_id, ses1, "dwi",
        f"{id}_{ses1}_dir-AP_run-1_part-mag_dwi.json"
    )

    json2 = os.path.join(
        rawdata, project_id, ses2, "dwi",
        f"{id}_{ses2}_dir-AP_run-1_part-mag_dwi.json"
    )

    with open(json1, "r") as f:
        meta1 = json.load(f)

    with open(json2, "r") as f:
        meta2 = json.load(f)

    t1 = datetime.strptime(meta1["AcquisitionTime"], "%H:%M:%S.%f")
    print(t1)
    t2 = datetime.strptime(meta2["AcquisitionTime"], "%H:%M:%S.%f")
    print(t2)

    delta_t_sec = (t1 - t2).total_seconds()
    delta_t = delta_t_sec / 60.0  # convert to minutes

    # ---- Compute time-normalized coefficient ----
    coeff = (data1 - data2) / delta_t
    coeff = np.nan_to_num(coeff)
    
    coeff_out=r"/home/malberti/Unix_Folders/SWEEP2/DEWEY/Time_coeff"
    os.makedirs(coeff_out, exist_ok=True)    # ---- Save NIfTI ----

    out_path = os.path.join(
        coeff_out,
        f"{id}_{set}_{metric}_timeNormCoeff.nii.gz"
    )
    nib.save(nib.Nifti1Image(coeff, img1.affine, img1.header), out_path)

    # ---- Mean coefficient (for plotting) ----
    coeff[(coeff > 2e-08) | (coeff < -2e-08)] = 0

    mean_val = np.mean(coeff)
    mean_coeffs.append(mean_val)

    print(f"{id} | Δt={delta_t:.1f}s | mean coeff={mean_val:.6e}")

# -----------------------------------
# Plot individual participant lines
# -----------------------------------

for vp, mean_val in zip(vps, mean_coeffs):

    x = [0, 1]
    y = [0, mean_val]

    plt.plot(x, y, marker="o")
    plt.text(1.02, mean_val, f"{vp:02d}", fontsize=7)

plt.xticks([0, 1], ["ses-01", "ses-02"])
plt.xlabel("Session")
plt.ylabel("Mean MD change per minute")
plt.title(f"{metric} Rate of Change | Set {set}")
plt.grid(True)

plt.tight_layout()
plt.show()